# Running Portfolio Performance

This notebook plots continuously compounded portfolio performance over every relevant observation date. The dates combine transaction dates and trading-price dates, following the same continuous-history approach as the portfolio value notebook.

For each interval ending on date `t`, the period performance is:

`((portfolio value at t - net I/O during the interval) / portfolio value at the previous observation) - 1`

The period returns are then compounded into a running performance. Inflows are removed from the ending value and outflows are added back, so contributions and withdrawals do not appear as investment performance. The lower panel keeps the inflow and outflow amounts visible next to the return series.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.core.services import RunningPerformanceService

In [ ]:
running_performance = RunningPerformanceService().calculate()

performance_columns = [
    'previous_date', 'date', 'start_value', 'inflow', 'outflow',
    'net_io', 'end_value', 'period_performance', 'performance',
]
performance_table = pd.DataFrame([
    {
        'previous_date': result.previous_date,
        'date': result.date,
        'start_value': result.start_value,
        'inflow': result.inflow,
        'outflow': result.outflow,
        'net_io': result.net_io,
        'end_value': result.end_value,
        'period_performance': result.period_performance,
        'performance': result.performance,
    }
    for result in running_performance
], columns=performance_columns)

print(f'Intervals with running performance: {len(performance_table)}')
display(performance_table)

In [ ]:
# The service returns Decimal values; convert only for chart rendering.
# The table above preserves the exact business values.
import matplotlib.pyplot as plt

chart_table = performance_table.copy()
if not chart_table.empty:
    chart_table['date'] = pd.to_datetime(chart_table['date'])
    chart_table['previous_date'] = pd.to_datetime(chart_table['previous_date'])

fig, (performance_axis, io_axis) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

performance_chart = chart_table.dropna(subset=['performance'])
if not performance_chart.empty:
    performance_axis.plot(
        performance_chart['date'],
        performance_chart['performance'].astype(float) * 100,
        color='#264653',
        linewidth=2,
        marker='o',
        markersize=3,
        label='Running performance',
    )
    performance_axis.legend()
performance_axis.axhline(0, color='#333333', linewidth=0.8)
performance_axis.set_title('Running Portfolio Performance')
performance_axis.set_ylabel('Performance (%)')
performance_axis.grid(axis='y', alpha=0.25)

if not chart_table.empty:
    io_dates = chart_table['date']
    io_axis.bar(
        io_dates,
        chart_table['inflow'].astype(float),
        color='#2a9d8f',
        alpha=0.75,
        label='Inflow',
    )
    io_axis.bar(
        io_dates,
        -chart_table['outflow'].astype(float),
        color='#e76f51',
        alpha=0.75,
        label='Outflow',
    )
    io_axis.legend()
io_axis.axhline(0, color='#333333', linewidth=0.8)
io_axis.set_title('I/O Used in Each Interval')
io_axis.set_xlabel('Date')
io_axis.set_ylabel('Amount')
io_axis.grid(axis='y', alpha=0.25)

plt.xticks(rotation=45)
fig.tight_layout()
plt.show()